# Análise do projeto

Exploração dos dados, comparação dos modelos e análise dos erros do melhor classificador.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from sentiment_analysis.data import load_splits, prepare_dataset
from sentiment_analysis.training import ClassicalPredictor

try:
    splits = load_splits('quick')
except FileNotFoundError:
    prepare_dataset('quick')
    splits = load_splits('quick')

In [ ]:
data = pd.concat(splits.values(), keys=splits, names=['split']).reset_index(level=0)
display(data.groupby(['split', 'label']).size().unstack(fill_value=0))
display(data.assign(caracteres=data.text.str.len()).groupby('label').caracteres.describe().round(1))

## Metodologia

Notas 1–2 viram `negative`, 3 vira `neutral` e 4–5 viram `positive`. O modo quick usa 1.200 textos por classe e divisão estratificada 64/16/20. O TF-IDF é ajustado somente no treino, os hiperparâmetros são escolhidos na validação e o teste permanece isolado.

In [ ]:
comparacao = pd.read_csv(ROOT / 'reports' / 'model_comparison.csv')
comparacao

## Análise de erros

A tabela de confusão e os exemplos abaixo são calculados diretamente no conjunto de teste. Assim, a análise pode ser reproduzida sem manter arquivos intermediários no Git.

In [ ]:
teste = splits['test'].copy()
preditor = ClassicalPredictor(ROOT / 'models' / 'best_classical_quick.joblib')
teste['prediction'] = [preditor.predict(texto).sentiment for texto in teste.text]

matriz = pd.crosstab(teste.label, teste.prediction, rownames=['Real'], colnames=['Predito'])
display(matriz)

erros = teste.loc[teste.label != teste.prediction, ['text', 'label', 'prediction']]
display(erros.groupby(['label', 'prediction']).size().rename('quantidade').reset_index())
display(erros.sample(n=min(12, len(erros)), random_state=42))

## Conclusões

A Regressão Logística oferece o melhor equilíbrio entre qualidade e latência. A classe neutra concentra os casos mais difíceis, principalmente em textos factuais curtos ou avaliações mistas. A avaliação GenAI usa uma amostra balanceada menor para controlar tempo e custo; por isso, seus números não devem ser comparados diretamente aos resultados de 720 itens sem observar a coluna de amostras.